<a href="https://colab.research.google.com/github/shadabcode09-sys/PUBLIC-TRANSPORT-ANALYSIS/blob/main/PROJECT_15.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [86]:
#SHADAB SHEIKH
#ADDMISSION NO:- 2026B15310390

import pandas as pd
import numpy as np

trips = pd.read_csv('trips.csv')
stops = pd.read_csv('stops.csv')
events = pd.read_csv('bus_events.csv')

print('Trips:', len(trips))
print('Stops:', len(stops))
print('Scheduled stop events:', len(events))

Trips: 6
Stops: 6
Scheduled stop events: 18


In [87]:
print('Trips table:')
print(trips.to_string(index=False))

print('\nFirst 6 bus events:')
print(events.head(6).to_string(index=False))

Trips table:
trip_id route_id service_date  capacity
    T01     R101   2026-11-12        40
    T02     R101   2026-11-12        40
    T03     R101   2026-11-14        40
    T04     R102   2026-11-13        35
    T05     R102   2026-11-13        35
    T06     R102   2026-11-15        35

First 6 bus events:
trip_id stop_id  stop_sequence scheduled_time actual_time  passenger_count service_alert
    T01     S01              1       08:00:00    08:00:00             22.0          none
    T01     S02              2       08:15:00    08:17:00             30.0          none
    T01     S03              3       08:30:00    08:33:00             35.0          none
    T02     S01              1       08:30:00    08:32:00             30.0          none
    T02     S02              2       08:45:00    08:48:00             37.0          none
    T02     S03              3       09:00:00    09:04:00             40.0          none


In [88]:
print('Duplicate trip IDs:', trips['trip_id'].duplicated().sum())
print('Duplicate stop IDs:', stops['stop_id'].duplicated().sum())
print('Unknown trip IDs:', (~events['trip_id'].isin(trips['trip_id'])).sum())

bus = events.merge(trips, on='trip_id', how='left')
bus = bus.merge(stops, on=['route_id', 'stop_id', 'stop_sequence'], how='left')
print('Wrong trip-stop links:', bus['stop_name'].isna().sum())

print('\nNumber of stop events in each trip:')
print(bus.groupby('trip_id')['stop_id'].count().to_string())
print('Trips with missing scheduled stop rows:', (bus.groupby('trip_id')['stop_id'].count() != 3).sum())

Duplicate trip IDs: 0
Duplicate stop IDs: 0
Unknown trip IDs: 0
Wrong trip-stop links: 0

Number of stop events in each trip:
trip_id
T01    3
T02    3
T03    3
T04    3
T05    3
T06    3
Trips with missing scheduled stop rows: 0


In [89]:
print('Duplicate event rows:', bus.duplicated(subset=['trip_id','stop_id','stop_sequence']).sum())
print('Missing actual arrivals:', bus['actual_time'].isna().sum())
print('Missing passenger counts:', bus['passenger_count'].isna().sum())

bus = bus.drop_duplicates(subset=['trip_id','stop_id','stop_sequence'])

bus['service_date'] = pd.to_datetime(bus['service_date'])
bus['scheduled_datetime'] = bus['service_date'] + pd.to_timedelta(bus['scheduled_time'])
bus['actual_datetime'] = bus['service_date'] + pd.to_timedelta(bus['actual_time'])

print('Scheduled times after midnight:', (bus['scheduled_time'].str[:2].astype(int) >= 24).sum())

Duplicate event rows: 0
Missing actual arrivals: 1
Missing passenger counts: 1
Scheduled times after midnight: 2


In [90]:
bus['delay_min'] = (bus['actual_datetime'] - bus['scheduled_datetime']).dt.total_seconds() / 60
bus['on_time'] = bus['delay_min'] <= 5

print(bus[['trip_id','stop_name','scheduled_time','actual_time','delay_min']].to_string(index=False))

valid = bus.dropna(subset=['delay_min'])
on_time_percent = valid['on_time'].mean() * 100
print('\nAverage delay:', round(valid['delay_min'].mean(), 2), 'minutes')
print('On-time percentage:', round(on_time_percent, 2), '%')

trip_id      stop_name scheduled_time actual_time  delay_min
    T01     North Gate       08:00:00    08:00:00        0.0
    T01        Library       08:15:00    08:17:00        2.0
    T01 Central Market       08:30:00    08:33:00        3.0
    T02     North Gate       08:30:00    08:32:00        2.0
    T02        Library       08:45:00    08:48:00        3.0
    T02 Central Market       09:00:00    09:04:00        4.0
    T03     North Gate       08:10:00    08:11:00        1.0
    T03        Library       08:25:00    08:27:00        2.0
    T03 Central Market       08:40:00         NaN        NaN
    T04     River Stop       13:00:00    13:08:00        8.0
    T04      Town Hall       13:15:00    13:25:00       10.0
    T04   Station Road       13:30:00    13:41:00       11.0
    T05     River Stop       13:30:00    13:39:00        9.0
    T05      Town Hall       13:45:00    13:57:00       12.0
    T05   Station Road       14:00:00    14:13:00       13.0
    T06     River Stop  

In [91]:
first_stops = bus[bus['stop_sequence'] == 1].copy()
first_stops = first_stops.sort_values(['route_id','service_date','actual_datetime'])

first_stops['headway_min'] = first_stops.groupby(['route_id','service_date'])['actual_datetime'].diff().dt.total_seconds() / 60
print(first_stops[['trip_id','route_id','service_date','headway_min']].to_string(index=False))
print('Average measured headway:', round(first_stops['headway_min'].mean(), 2), 'minutes')

trip_id route_id service_date  headway_min
    T01     R101   2026-11-12          NaN
    T02     R101   2026-11-12         32.0
    T03     R101   2026-11-14          NaN
    T04     R102   2026-11-13          NaN
    T05     R102   2026-11-13         31.0
    T06     R102   2026-11-15          NaN
Average measured headway: 31.5 minutes


In [92]:
delays = valid['delay_min'].to_numpy()
print('90th percentile delay:', round(np.percentile(delays, 90), 2), 'minutes')
print('Delay standard deviation:', round(np.std(delays), 2), 'minutes')

known_loads = bus['passenger_count'].dropna().to_numpy()
print('Maximum passenger load:', int(np.max(known_loads)), 'passengers')

bus['load_percent'] = bus['passenger_count'] / bus['capacity'] * 100
print('Highest capacity usage:', round(np.max(bus['load_percent'].dropna().to_numpy()), 2), '%')

90th percentile delay: 12.4 minutes
Delay standard deviation: 4.69 minutes
Maximum passenger load: 40 passengers
Highest capacity usage: 100.0 %


In [93]:
bus['day_type'] = np.where(bus['service_date'].dt.dayofweek < 5, 'Weekday', 'Weekend')

hour = bus['scheduled_time'].str[:2].astype(int) % 24
bus['time_period'] = np.where(((hour >= 7) & (hour < 10)) | ((hour >= 17) & (hour < 20)), 'Peak', 'Off-peak')

print('Weekday vs weekend:')
print(bus.groupby('day_type')['passenger_count'].agg(['count','mean']).round(2).to_string())

print('\nPeak vs off-peak:')
print(bus.groupby('time_period')['passenger_count'].agg(['count','mean']).round(2).to_string())

print('\nDay type and period:')
print(bus.groupby(['day_type','time_period'])['passenger_count'].mean().round(2).to_string())

Weekday vs weekend:
          count   mean
day_type              
Weekday      11  30.73
Weekend       6  17.83

Peak vs off-peak:
             count   mean
time_period              
Off-peak         8  23.88
Peak             9  28.22

Day type and period:
day_type  time_period
Weekday   Off-peak       28.80
          Peak           32.33
Weekend   Off-peak       15.67
          Peak           20.00


In [94]:
route_summary = bus.groupby('route_id').agg(
    total_trips=('trip_id','nunique'),
    recorded_arrivals=('delay_min','count'),
    average_delay=('delay_min','mean'),
    on_time_percent=('on_time','mean')
).reset_index()

on_time_by_route = valid.groupby('route_id')['on_time'].mean() * 100
route_summary['on_time_percent'] = route_summary['route_id'].map(on_time_by_route)
route_summary['on_time_percent'] = route_summary['on_time_percent'].round(2)
route_summary['average_delay'] = route_summary['average_delay'].round(2)

print('Route reliability:')
print(route_summary.to_string(index=False))

print('\nStop reliability:')
stop_summary = bus.groupby(['route_id','stop_name'])['delay_min'].mean().round(2).reset_index()
stop_summary = stop_summary.sort_values('delay_min', ascending=False)
print(stop_summary.to_string(index=False))

priority_routes = route_summary.sort_values('average_delay', ascending=False)
print('\nPriority routes (highest delay first):')
print(priority_routes[['route_id','average_delay','on_time_percent']].to_string(index=False))

Route reliability:
route_id  total_trips  recorded_arrivals  average_delay  on_time_percent
    R101            3                  8           2.12            100.0
    R102            3                  9          11.00              0.0

Stop reliability:
route_id      stop_name  delay_min
    R102   Station Road      12.67
    R102      Town Hall      11.33
    R102     River Stop       9.00
    R101 Central Market       3.50
    R101        Library       2.33
    R101     North Gate       1.00

Priority routes (highest delay first):
route_id  average_delay  on_time_percent
    R102          11.00              0.0
    R101           2.12            100.0


In [95]:
trip_delays = bus.groupby(['route_id','trip_id'])['delay_min'].mean().reset_index()
trip_delays['late_trip'] = trip_delays['delay_min'] > 5
repeated = trip_delays.groupby('route_id')['late_trip'].sum().reset_index()
repeated['repeated_issue'] = repeated['late_trip'] >= 2
print(repeated.to_string(index=False))

route_id  late_trip  repeated_issue
    R101          0           False
    R102          3            True


In [96]:
r101 = bus[bus['route_id'] == 'R101']
max_load = r101['passenger_count'].max()
old_capacity = 40
new_capacity = 50
old_usage = max_load / old_capacity * 100
new_usage = max_load / new_capacity * 100

print('Maximum load on R101:', int(max_load))
print('Old capacity usage:', round(old_usage, 2), '%')
print('New capacity usage:', round(new_usage, 2), '%')

Maximum load on R101: 40
Old capacity usage: 100.0 %
New capacity usage: 80.0 %


In [97]:
total_events = len(bus)
known_counts = bus['passenger_count'].notna().sum()
count_coverage = known_counts / total_events * 100

print('Scheduled stop events:', total_events)
print('Available passenger counts:', known_counts)
print('Passenger count coverage:', round(count_coverage, 2), '%')
print('Missing arrival events:', bus['actual_time'].isna().sum())
print('Service alerts (sample labels):')
print(bus['service_alert'].value_counts().to_string())

Scheduled stop events: 18
Available passenger counts: 17
Passenger count coverage: 94.44 %
Missing arrival events: 1
Service alerts (sample labels):
service_alert
none         9
traffic      6
road_work    3


In [98]:
bus.to_csv('validated_bus_events.csv', index=False)
route_summary.to_csv('route_reliability.csv', index=False)
priority_routes.to_csv('priority_routes.csv', index=False)
print('Saved: validated_bus_events.csv, route_reliability.csv, priority_routes.csv')

Saved: validated_bus_events.csv, route_reliability.csv, priority_routes.csv
